In [2]:

!pip -q install google-genai

from google import genai
from google.colab import userdata
import sqlite3

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

DB_NAME = "/content/interview.db"

conn = sqlite3.connect(DB_NAME)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS employees (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    department TEXT,
    salary INTEGER
)
""")

cursor.execute("DELETE FROM employees")

cursor.executemany("""
INSERT INTO employees (name, department, salary)
VALUES (?, ?, ?)
""", [
    ("Rahul", "IT", 60000),
    ("Priya", "HR", 45000),
    ("Anu", "IT", 75000),
    ("Kiran", "Finance", 50000),
    ("Meera", "HR", 65000)
])

conn.commit()
conn.close()

print("Gemini initialized and sample database created!")


Gemini initialized and sample database created!


In [3]:

SYSTEM_PROMPT = """
You are an expert SQL interview assistant.

Database schema:
employees(id, name, department, salary)

For each user question:
1. Answer SQL theory questions clearly with examples.
2. For database questions, generate one SQLite SELECT query.
3. Explain what the query does.
4. Use only the employees table and its listed columns.
5. Return SQL between the exact markers:
   SQL_START
   your SQL query
   SQL_END
6. Never generate INSERT, UPDATE, DELETE, DROP,
   ALTER, CREATE, or multiple SQL statements.
7. Never invent query results. Actual results come from execution.
8. If the question is general SQL theory and does not require
   database execution, answer normally without SQL markers.
"""

def ask_sql_ai(question):
    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=SYSTEM_PROMPT + "\n\nUser question:\n" + question
    )
    return response.text or "No response returned."

print("SQL AI assistant is ready!")


SQL AI assistant is ready!


In [7]:

import time

# Try alternative models available in your API catalogue.
MODEL_CANDIDATES = [
    "gemini-flash-lite-latest",
    "gemini-3.1-flash-lite",
    "gemini-3.5-flash",
    "gemini-flash-latest",
    "gemini-3.8-flash",
    "gemini-3.7-flash",
    "gemini-3.6-flash",
    "gemini-3-flash-preview"
]

def ask_sql_ai(question):
    errors = []

    for model_name in MODEL_CANDIDATES:
        try:
            response = client.models.generate_content(
                model=model_name,
                contents=(
                    SYSTEM_PROMPT
                    + "\n\nUser question:\n"
                    + question
                )
            )

            if response.text:
                print("Model used:", model_name)
                return response.text

            errors.append(f"{model_name}: Empty response")

        except Exception as error:
            errors.append(f"{model_name}: {error}")
            print(f"{model_name} failed; trying another model.")

    raise RuntimeError(
        "No model returned a response. Check API availability "
        "and quota.\n" + "\n".join(errors)
    )


In [8]:

prompt = input("Enter your SQL interview question: ")

try:
    process_sql_question(prompt)
except Exception as error:
    print("\nAI service unavailable:", error)
    print("Please try again later if all models fail.")


Enter your SQL interview question: explain primary key with example
Model used: gemini-flash-lite-latest
AI Response:

A **Primary Key** is a column (or a set of columns) in a table that uniquely identifies each row in that table. 

### Key Characteristics of a Primary Key:
1. **Uniqueness:** No two rows can have the same value in the primary key column.
2. **Not Null:** A primary key column cannot contain `NULL` values. Every row must have a valid value.
3. **Uniqueness Constraint per Table:** A table can have only one primary key (though that primary key can consist of multiple columns, known as a composite primary key).

---

### Example:

Consider our `employees` table schema:
- `id` (Primary Key)
- `name`
- `department`
- `salary`

| id | name | department | salary |
|----|------|------------|--------|
| 1  | Alice| HR         | 60000  |
| 2  | Bob  | IT         | 80000  |
| 3  | Charlie | Finance | 75000  |

In this example, the `id` column acts as the **Primary Key**. 
- You can